In [ ]:
import os
import glob
import pandas as pd

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
softmax_dir = r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\model\model_fits(fixed alpha)\softmax\neg_llh_alpha_1"

result_df_path = r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\data\result_df\all_result_df_with_behavior_summaries.csv"

output_path = r"C:\Users\zyfxl\OneDrive - UC Irvine\UCI\CCNL Lab\Foraging Task Overview\foraging_MQLP-main\foraging_MQLP-main\data\result_df\merged_results_with_params.csv"

# --------------------------------------------------
# 2. List files — diagnostic
# --------------------------------------------------
print("Softmax directory:", softmax_dir)
print("Directory exists:", os.path.exists(softmax_dir))

all_files = glob.glob(os.path.join(softmax_dir, "*"))
print("\nAll files found:")
for f in all_files:
    print("  ", f)

# Your files look like: sub0b2172zv.csv, sub41ns7le1.csv, ...
files = glob.glob(os.path.join(softmax_dir, "sub*.csv"))
print("\nSoftmax CSV files (sub*.csv):")
for f in files:
    print("  ", f)

if len(files) == 0:
    raise ValueError("❌ No 'sub*.csv' files found. Check folder or naming.")

# --------------------------------------------------
# 3. Load softmax fits
# --------------------------------------------------
softmax_rows = []

for f in files:
    try:
        df = pd.read_csv(f)

        # Must contain subject_num column
        if "subject_num" not in df.columns:
            raise ValueError(f"{f} missing 'subject_num' column")

        row = df.iloc[0].copy()
        subject_num = str(row["subject_num"])

        # Create merge key used by result_df
        row["subject_num_fileID"] = subject_num

        softmax_rows.append(row)

    except Exception as e:
        print(f"⚠️ Error loading {f}: {e}")

if len(softmax_rows) == 0:
    raise ValueError("❌ No valid softmax data loaded.")

softmax_all = pd.DataFrame(softmax_rows)

print("\nLoaded softmax rows:")
print(softmax_all.head())

# --------------------------------------------------
# 4. Load result_df
# --------------------------------------------------
result_df = pd.read_csv(result_df_path)

if "subject_num_fileID" not in result_df.columns:
    raise ValueError("❌ result_df does NOT contain 'subject_num_fileID' column!")

# --------------------------------------------------
# 5. Merge (left merge → retain all subjects)
# --------------------------------------------------
merged = result_df.merge(
    softmax_all,
    on="subject_num_fileID",
    how="left"
)

# --------------------------------------------------
# 6. Save merged data
# --------------------------------------------------
merged.to_csv(output_path, index=False)

print("\n🎉 Merge completed successfully!")
print("Saved merged file →", output_path)
print("Merged dataframe shape:", merged.shape)
